<a href="https://colab.research.google.com/github/Nathruth/mlcamp-homework/blob/main/deeplearninghw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: basic setup & reproducibility
import os
import random
import numpy as np
import torch

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)


Device: cuda


In [2]:
# Cell 2: shell cell - downloads and unzips dataset
# (run in Colab using a code cell with prefix `!`)
!wget -O data.zip "https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip"
!unzip -q data.zip -d data
!ls -la data


--2025-11-30 14:24:55--  https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/405934815/e712cf72-f851-44e0-9c05-e711624af985?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-11-30T14%3A58%3A30Z&rscd=attachment%3B+filename%3Ddata.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-11-30T13%3A57%3A58Z&ske=2025-11-30T14%3A58%3A30Z&sks=b&skv=2018-11-09&sig=yXk6X9BeUJy%2FnsSF%2BTKG1VCkzjbErdwcvqgB7A%2B0%2BCs%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NDUxNDQ5NiwibmJmIjoxNzY0NTEyNjk2LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdG

In [25]:
# run this to confirm the layout you described
import os
for root, dirs, files in os.walk("data"):
    print(root)
    # print only a few entries so output stays readable
    print("  dirs:", dirs[:10])
    print("  files (sample):", files[:10])
    print("---")



data
  dirs: ['data']
  files (sample): []
---
data/data
  dirs: ['test', 'train']
  files (sample): []
---
data/data/test
  dirs: ['curly', 'straight']
  files (sample): []
---
data/data/test/curly
  dirs: []
  files (sample): ['indian-hairstyles-for-short-hair-1.jpg', 'image94.jpg', 'Curls-1600x900.jpg', 'image2.jpg', 'image87.jpg', 'image170.jpg', 'Untitled-1.jpg', 'Medium-Curly-Hairstyles-for-2018-2019.jpg', 'images506.jpg', 'images15.jpg']
---
data/data/test/straight
  dirs: []
  files (sample): ['images201.jpg', 'caf03e2b33942738274b9e3d44632040.jpg', 'images611.jpg', 'hp8302-04.jpg', 'e374f7844815aaeb27d48dac10c5e9ff.jpg', 'images602.jpg', 'image17.jpeg', 'images221.jpg', '6ede32dd8dfb38486b88c582c4acbe75.jpg', 'IMG_6585.PNG']
---
data/data/train
  dirs: ['curly', 'straight']
  files (sample): []
---
data/data/train/curly
  dirs: []
  files (sample): ['image230.jpg', 'images201.jpg', 'image216.jpg', 'baddie-selfie-curly-hair-Favim.com-8.jpg', 'IMG_0391-570x350.jpg', 'image226.jp

In [26]:
# transforms (same as before)
from torchvision import transforms, datasets
from torch.utils.data import random_split, DataLoader

transform_train = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # ImageNet normalization
])

transform_eval = transforms.Compose([
    transforms.Resize((200,200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# Paths (adjust only if your unzip produced a different inner folder name)
TRAIN_ROOT = "data/data/train"
TEST_ROOT  = "data/data/test"

train_full = datasets.ImageFolder(root=TRAIN_ROOT, transform=transform_train)
test_set   = datasets.ImageFolder(root=TEST_ROOT,  transform=transform_eval)

print("Classes (train):", train_full.classes)
print("Total train images:", len(train_full))
print("Total test images:", len(test_set))

# show counts per class in train
from collections import Counter
train_targets = [y for _, y in train_full.imgs]   # imgs is list of (path, class_idx)
print("Train class counts:", Counter(train_targets))

# split train -> train/val
SEED = 42
n = len(train_full)
n_train = int(n * 0.8)
n_val   = n - n_train
generator = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(train_full, [n_train, n_val], generator=generator)

# Important: random_split keeps the original Dataset object; val_ds will use transform_train.
# To use transform_eval for validation, wrap subsets into new datasets:
from torch.utils.data import Subset
# If you want val to use eval transforms (no augmentation), create a copy of ImageFolder with eval transforms:
train_full_noaug = datasets.ImageFolder(root=TRAIN_ROOT, transform=transform_eval)
val_ds = Subset(train_full_noaug, val_ds.indices)  # val uses eval transforms
train_ds = Subset(train_full, train_ds.indices)    # train keeps augmentation

batch_size = 20
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("Batches - train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))



Classes (train): ['curly', 'straight']
Total train images: 800
Total test images: 201
Train class counts: Counter({0: 410, 1: 390})
Batches - train: 32 val: 8 test: 11


In [27]:
# Cell 5a: recommended model (no sigmoid, use BCEWithLogitsLoss)
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # input shape (3, 200, 200)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=(3,3), padding=1)  # keeps size
        self.pool = nn.MaxPool2d(kernel_size=(2,2))
        # after conv + pool: 32 channels, size 100x100
        # flatten, then linear  -> need to compute flattened size
        self.flatten_dim = 32 * 100 * 100
        self.fc1 = nn.Linear(self.flatten_dim, 64)
        self.fc_out = nn.Linear(64, 1)  # logits

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)  # flatten
        x = F.relu(self.fc1(x))
        x = self.fc_out(x)         # NOTE: no sigmoid here if using BCEWithLogitsLoss
        return x

model = SimpleCNN().to(device)
print(model)


SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=320000, out_features=64, bias=True)
  (fc_out): Linear(in_features=64, out_features=1, bias=True)
)


In [28]:
# Cell 6: criterion and optimizer
# Recommended: BCEWithLogitsLoss with model (SimpleCNN) that returns logits
criterion = nn.BCEWithLogitsLoss()   # expects raw logits; more stable

optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)


In [29]:
from torchsummary import summary
summary(model, input_size=(3, 200, 200))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 200, 200]             896
         MaxPool2d-2         [-1, 32, 100, 100]               0
            Linear-3                   [-1, 64]      20,480,064
            Linear-4                    [-1, 1]              65
Total params: 20,481,025
Trainable params: 20,481,025
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 12.21
Params size (MB): 78.13
Estimated Total Size (MB): 90.79
----------------------------------------------------------------


In [30]:
# --- make sure your DataLoaders were created like this earlier ---
# train_loader = DataLoader(train_ds, batch_size=20, shuffle=True,  num_workers=2, pin_memory=True)
# val_loader   = DataLoader(val_ds,   batch_size=20, shuffle=False, num_workers=2, pin_memory=True)

# Recommended: BCEWithLogitsLoss if model returns logits
# criterion = nn.BCEWithLogitsLoss()
# optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:   # train_loader yields batches already
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)  # shape (N,1)

        optimizer.zero_grad()
        outputs = model(images)               # logits if using BCEWithLogitsLoss
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)   # accumulate sum loss
        preds = (torch.sigmoid(outputs) > 0.5).float() # threshold logits -> probs -> preds
        total_train += labels.size(0)
        correct_train += (preds == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    # Validation
    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (preds == labels).sum().item()

    val_epoch_loss = val_running_loss / len(val_loader.dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f} | "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

    # optional: save best model by val loss
    # if epoch == 0 or val_epoch_loss < best_val_loss:
    #     torch.save(model.state_dict(), "best_model.pth")
    #     best_val_loss = val_epoch_loss


Epoch 1/10 | Train Loss: 0.6730, Train Acc: 0.6188 | Val Loss: 0.6196, Val Acc: 0.6312
Epoch 2/10 | Train Loss: 0.5669, Train Acc: 0.6859 | Val Loss: 0.6022, Val Acc: 0.6750
Epoch 3/10 | Train Loss: 0.5068, Train Acc: 0.7391 | Val Loss: 0.6313, Val Acc: 0.6500
Epoch 4/10 | Train Loss: 0.4551, Train Acc: 0.7937 | Val Loss: 0.5797, Val Acc: 0.7063
Epoch 5/10 | Train Loss: 0.3801, Train Acc: 0.8219 | Val Loss: 0.6230, Val Acc: 0.6750
Epoch 6/10 | Train Loss: 0.3055, Train Acc: 0.8641 | Val Loss: 0.5892, Val Acc: 0.7188
Epoch 7/10 | Train Loss: 0.2482, Train Acc: 0.9141 | Val Loss: 0.6936, Val Acc: 0.7250
Epoch 8/10 | Train Loss: 0.2316, Train Acc: 0.9078 | Val Loss: 1.2015, Val Acc: 0.7000
Epoch 9/10 | Train Loss: 0.1817, Train Acc: 0.9281 | Val Loss: 0.6618, Val Acc: 0.7000
Epoch 10/10 | Train Loss: 0.0843, Train Acc: 0.9828 | Val Loss: 0.7538, Val Acc: 0.7500


In [31]:
import numpy as np
from torchvision import transforms, datasets
from torch.utils.data import Subset, DataLoader

# 1) Define the augmentation transform (RandomResizedCrop first, then rotation & flip)
augment_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomRotation(50),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# 2) Keep the validation/test transforms as before (no augmentation)
eval_transform = transforms.Compose([
    transforms.Resize((200,200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# 3) Re-create ImageFolder datasets for train (with augmentation) and val/test (no aug)
# IMPORTANT: do not change the train/val split indices used previously.
# If earlier you created random_split, you should have the indices available.
# I'll assume you still have `train_ds` and `val_ds` as Subset objects OR you have `train_indices` and `val_indices`.
# If you only have train_ds and val_ds as Subsets from before, extract indices like this:
try:
    train_indices = train_ds.indices
    val_indices = val_ds.indices
except Exception:
    # if train_ds is not a Subset but a Dataset, this will fail; in that case you should have created
    # train_ds,val_ds earlier via random_split and can recreate indices similarly. If not, re-run the split.
    raise RuntimeError("Could not find train_ds.indices and val_ds.indices. Make sure you kept the split indices.")

# create new ImageFolder instances pointing to the same root
TRAIN_ROOT = "data/data/train"   # adjust if needed
train_full_aug = datasets.ImageFolder(root=TRAIN_ROOT, transform=augment_train_transform)
train_full_noaug = datasets.ImageFolder(root=TRAIN_ROOT, transform=eval_transform)  # for val

# build Subsets with same indices so the split stays identical
train_subset_aug = Subset(train_full_aug, train_indices)
val_subset_noaug  = Subset(train_full_noaug, val_indices)

# 4) Recreate DataLoaders (batch_size = 20 as requested)
batch_size = 20
train_loader = DataLoader(train_subset_aug, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_subset_noaug,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
# test_loader should already exist (from earlier). If not, recreate it:
# TEST_ROOT = "data/data/test"
# test_set = datasets.ImageFolder(root=TEST_ROOT, transform=eval_transform)
# test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)

# 5) Continue training the SAME model for 10 more epochs (do NOT re-create the model or optimizer)
num_epochs = 10
test_losses = []            # will collect test loss after each epoch
# history augmentation continuation (optional)
history_aug = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    # ---- training (same loop as before) ----
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (preds == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct_train / total_train
    history_aug['loss'].append(epoch_loss)
    history_aug['acc'].append(epoch_acc)

    # ---- validation ----
    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (preds == labels).sum().item()

    val_epoch_loss = val_running_loss / len(val_loader.dataset)
    val_epoch_acc = correct_val / total_val
    history_aug['val_loss'].append(val_epoch_loss)
    history_aug['val_acc'].append(val_epoch_acc)

    # ---- test evaluation after this epoch (collect test loss) ----
    test_running_loss = 0.0
    total_test = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            test_running_loss += loss.item() * images.size(0)
            total_test += labels.size(0)

    test_epoch_loss = test_running_loss / total_test
    test_losses.append(test_epoch_loss)

    print(f"(Aug) Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f} | "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f} | "
          f"Test Loss: {test_epoch_loss:.4f}")

# 6) After training, compute mean test loss across all epochs
mean_test_loss = np.mean(test_losses)
print("Mean test loss across epochs (augmentations):", mean_test_loss)


(Aug) Epoch 1/10 | Train Loss: 0.7297, Train Acc: 0.6344 | Val Loss: 0.7115, Val Acc: 0.6438 | Test Loss: 0.6572
(Aug) Epoch 2/10 | Train Loss: 0.5933, Train Acc: 0.6844 | Val Loss: 0.6586, Val Acc: 0.6687 | Test Loss: 0.6102
(Aug) Epoch 3/10 | Train Loss: 0.5557, Train Acc: 0.7094 | Val Loss: 0.6474, Val Acc: 0.6625 | Test Loss: 0.6152
(Aug) Epoch 4/10 | Train Loss: 0.5542, Train Acc: 0.7141 | Val Loss: 0.6075, Val Acc: 0.7125 | Test Loss: 0.6012
(Aug) Epoch 5/10 | Train Loss: 0.5197, Train Acc: 0.7250 | Val Loss: 0.6347, Val Acc: 0.6875 | Test Loss: 0.6231
(Aug) Epoch 6/10 | Train Loss: 0.4951, Train Acc: 0.7391 | Val Loss: 0.6427, Val Acc: 0.7063 | Test Loss: 0.5933
(Aug) Epoch 7/10 | Train Loss: 0.5056, Train Acc: 0.7312 | Val Loss: 0.6001, Val Acc: 0.6750 | Test Loss: 0.5750
(Aug) Epoch 8/10 | Train Loss: 0.4936, Train Acc: 0.7500 | Val Loss: 0.5613, Val Acc: 0.7375 | Test Loss: 0.5520
(Aug) Epoch 9/10 | Train Loss: 0.4690, Train Acc: 0.7828 | Val Loss: 0.7204, Val Acc: 0.6875 | T

In [32]:
print("Model last layer:", model.fc_out if hasattr(model, "fc_out") else list(model.children())[-1])
print("Criterion:", criterion)


Model last layer: Linear(in_features=64, out_features=1, bias=True)
Criterion: BCEWithLogitsLoss()


In [33]:
import torch

def compute_test_accuracy(loader, model, device, use_sigmoid=False):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            outputs = model(images)
            if use_sigmoid:
                outputs = torch.sigmoid(outputs)  # convert logits to probabilities

            preds = (outputs > 0.5).float()      # threshold at 0.5
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    return acc

# --- Usage ---
# 1) Test accuracy for your current model (logits)
acc_logits = compute_test_accuracy(test_loader, model, device, use_sigmoid=False)
print("Test accuracy (logits, no sigmoid applied):", acc_logits)

# 2) Test accuracy assuming model outputs probabilities (sigmoid)
acc_sigmoid = compute_test_accuracy(test_loader, model, device, use_sigmoid=True)
print("Test accuracy (with sigmoid applied):", acc_sigmoid)


Test accuracy (logits, no sigmoid applied): 0.7014925373134329
Test accuracy (with sigmoid applied): 0.7014925373134329


In [34]:
# 1) Add Sigmoid to model output
model_with_sigmoid = nn.Sequential(model, nn.Sigmoid())

# 2) Use BCELoss
criterion = nn.BCELoss()

# 3) Compute test loss on test_loader
test_loss_list = []
model_with_sigmoid.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        outputs = model_with_sigmoid(images)
        loss = criterion(outputs, labels)
        test_loss_list.append(loss.item())

mean_test_loss = sum(test_loss_list) / len(test_loss_list)
print("Mean test loss (probabilities + BCELoss):", mean_test_loss)


Mean test loss (probabilities + BCELoss): 0.6024179986932061
